# Compressed KV cache and matrix absorption

This notebook compares ordinary full-sequence MLA with absorbed one-token decoding using the same weights.

In [ ]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(14)

In [ ]:
from src.model import CachedMLA, DeepSeekConfig

config = DeepSeekConfig(
    dim=64, n_heads=4, q_lora_rank=16, kv_lora_rank=8,
    qk_nope_head_dim=8, qk_rope_head_dim=4, v_head_dim=8,
    max_seq_len=32,
)
model = CachedMLA(config).eval()
hidden_states = torch.randn(1, 6, config.dim)

In [ ]:
with torch.no_grad():
    naive = model(hidden_states)
    model.reset_cache()
    model.prefill(hidden_states[:, :-1])
    absorbed_last = model.decode_step(hidden_states[:, -1:])

print("naive output:", naive.shape)
print("absorbed output:", absorbed_last.shape)
print("maximum error:", (naive[:, -1:] - absorbed_last).abs().max().item())
print("latent cache:", model.kv_cache.shape)
print("position cache:", model.pe_cache.shape)

## Decode every token incrementally

The cache starts empty and grows by one position per call.

In [ ]:
model.reset_cache()
outputs = []
with torch.no_grad():
    for position in range(hidden_states.shape[1]):
        outputs.append(model.decode_step(hidden_states[:, position:position + 1]))
incremental = torch.cat(outputs, dim=1)

torch.testing.assert_close(incremental, naive, atol=2e-4, rtol=2e-4)
print("incremental equivalence passed")
print("cached positions:", model.cache_length)

In [ ]:
def tensor_mib(tensor):
    return tensor.numel() * tensor.element_size() / (1024 ** 2)

print("latent dtype:", model.kv_cache.dtype)
print("latent MiB:", tensor_mib(model.kv_cache))
print("position MiB:", tensor_mib(model.pe_cache))